# Bernstein-Vazirani
The oracle hides a secret string $s$ via $f(x)=s\cdot x \bmod 2$. One query recovers all $n$ bits of $s$ at once.
Measure the input register: the bitstring you read out **is** $s$.
Runs on the local Aer simulator (noiseless).

In [ ]:
# Oracle: CX from input qubit i into the target wherever secret bit s_i = 1
from qiskit import QuantumCircuit

def bv_oracle(s: str) -> QuantumCircuit:
    n = len(s)
    oracle = QuantumCircuit(n + 1)
    for i, bit in enumerate(reversed(s)):   # qubit 0 = least-significant bit
        if bit == "1":
            oracle.cx(i, n)
    return oracle

In [ ]:
# Assemble the full Bernstein-Vazirani circuit and draw it
s = "1011"                    # the secret to recover
n = len(s)

qc = QuantumCircuit(n + 1, n)
qc.x(n)                       # target starts in |1>
qc.h(range(n + 1))            # superposition + |-> on target
qc.barrier()
qc.compose(bv_oracle(s), inplace=True)
qc.barrier()
qc.h(range(n))               # interference collapses the input register onto s
qc.measure(range(n), range(n))

qc.draw("mpl")

In [ ]:
# Run on the Aer simulator
from qiskit_aer import AerSimulator

counts = AerSimulator().run(qc, shots=1024).result().get_counts()

In [ ]:
# The dominant bitstring should equal s
from qiskit.visualization import plot_histogram

print("secret s      :", s)
print("most frequent :", max(counts, key=counts.get))
plot_histogram(counts)